## MIE1624H Course Project Notebook
This notebook is part of our final course project submission. It has been cleaned to comply with submission requirements:
- API keys have been removed.
- API calls and installation steps are commented out and separated where applicable.
- All valid data processing code is retained and executable.
- Notebook is Google Colab compatible.
- Referenced data files are included in the final submission folder.

### Commented API/Installation Code (Submission Version)

In [ ]:
# (Commented out for submission)
# !pip install google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2
# (Commented out for submission)
# import openai
# (Commented out for submission)
# client = openai.OpenAI(api_key=OPENAI_API_KEY)
# (Commented out for submission)
# # 🔹 Define search queries for different countries
# (Commented out for submission)
#     "Sweden": ["Sweden R&D spending", "Sweden AI research funding"],
# (Commented out for submission)
#     "India": ["India tech startups", "India AI research", "India digital economy"],
# (Commented out for submission)
# def search_youtube_videos():
# (Commented out for submission)
#     youtube = googleapiclient.discovery.build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
# (Commented out for submission)
#             request = youtube.search().list(
# (Commented out for submission)
#                     "video_url": f"https://www.youtube.com/watch?v={item['id']['videoId']}"
# (Commented out for submission)
#     url = f"https://www.youtube.com/watch?v={video_id}"
# (Commented out for submission)
#     videos = search_youtube_videos()
# (Commented out for submission)
#         print("❌ [ERROR] No relevant videos retrieved! Adjust search queries.")

In [ ]:
import requests
import googleapiclient.discovery
import googleapiclient.errors
import fitz  # PyMuPDF for PDFs
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from docx import Document
import numpy as np
from datetime import datetime
import os

# 🔹 API Keys (Replace with your own)
YOUTUBE_API_KEY = "your_youtubeapi_key_here"
OPENAI_API_KEY = "your_openai_key_here"

# 🔹 Initialize OpenAI client

COUNTRY_QUERIES = {
    "Canada": ["Canada AI policy", "Canadian innovation ecosystem", "Canada startup funding"],
    "USA": ["USA AI regulation", "US innovation policies", "Silicon Valley tech growth"],
    "China": ["China AI policies", "China tech innovation", "China startup ecosystem"],
    "Germany": ["Germany industrial innovation", "Germany AI strategy"],
    "United Kingdom": ["UK AI regulation", "UK innovation policies", "UK startup funding"],
    "England": ["England AI policies", "England R&D investments"]
}

# 🔹 Search for relevant YouTube videos
    videos = []

    for country, queries in COUNTRY_QUERIES.items():
        for query in queries:
                q=query,
                part="snippet",
                type="video",
                maxResults=5,  # Fetch 5 videos per query
                relevanceLanguage="en"
            )
            response = request.execute()

            for item in response["items"]:
                videos.append({
                    "country": country,
                    "video_id": item["id"]["videoId"],
                    "title": item["snippet"]["title"],
                    "description": item["snippet"]["description"],
                    "published_at": item["snippet"]["publishedAt"],
                    "channel_title": item["snippet"]["channelTitle"],
                })

    print(f"✅ [DEBUG] Retrieved {len(videos)} relevant YouTube videos.")
    return videos

# 🔹 Retrieve video transcripts using OpenAI Speech-to-Text
def get_video_transcript(video_id):
    
    try:
        transcript_response = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": f"Extract and summarize transcript from: {url}"}]
        )
        return transcript_response.choices[0].message.content.strip()
    except Exception as e:
        return "Transcript unavailable."

# 🔹 Topic classification & clustering
def classify_topics(video_descriptions):
    vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
    X = vectorizer.fit_transform(video_descriptions)
    
    num_clusters = 5  # Define number of clusters
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    kmeans.fit(X)
    
    return kmeans.labels_

# 🔹 Main Execution
def main():
    if not videos:
        return

    summarized_videos = []
    descriptions = []
    
    for video in videos:
        transcript = get_video_transcript(video["video_id"])
        descriptions.append(video["description"])

        summarized_videos.append({
            "country": video["country"],
            "title": video["title"],
            "summary": transcript,
            "published_at": video["published_at"],
            "video_url": video["video_url"],
            "channel": video["channel_title"]
        })

    topic_labels = classify_topics(descriptions)

    # 🔹 Append detailed video aggregation results to the Appendix
    APPENDIX_SUMMARIES = {}
    for idx, video in enumerate(summarized_videos):
        country = video["country"]
        if country not in APPENDIX_SUMMARIES:
            APPENDIX_SUMMARIES[country] = []
        APPENDIX_SUMMARIES[country].append(f"🔹 {video['title']}")

    # 🔹 Create and structure the Word document
    doc = Document()
    doc.add_heading("Canadian Innovation Policy - YouTube Video Analysis", level=1)
    doc.add_paragraph(f"Generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # 🔹 Add Summarized Videos to the Document
    doc.add_heading("YouTube Innovation Video Summaries", level=2)
    for video in summarized_videos:
        doc.add_heading(f"{video['country']} - {video['title']}", level=3)
        doc.add_paragraph(video["summary"])
        doc.add_paragraph(f"🔹 Published Date: {video['published_at']}")
        doc.add_paragraph(f"🔹 Watch: {video['video_url']}")
        doc.add_paragraph(f"🔹 Channel: {video['channel']}")

    # 🔹 Add Appendix
    doc.add_page_break()
    doc.add_heading("Appendix A: YouTube Video Aggregation Summaries", level=2)
    for country, videos in APPENDIX_SUMMARIES.items():
        doc.add_heading(f"{country} - Innovation Videos", level=3)
        for video_title in videos:
            doc.add_paragraph(video_title)

    # 🔹 Save the document
    report_path = "YouTubeAPI_Canadian_Innovation_Strategy.docx"
    doc.save(report_path)

    print(f"✅ [SUCCESS] Saved Report: {report_path}")

# Run the script
if __name__ == "__main__":
    main()